# CRIVO — Medir antes de treinar

Este caderno **não treina nada**. Ele mede o ponto de partida:
1. confere os exemplos gerados no Drive (quantos estão `ok`, quantos precisam de revisão);
2. roda as réguas de compreensão no Crivo atual e mostra o resultado.

Use CPU (o padrão do Colab já serve). Leva alguns minutos.

In [ ]:
# Célula 0 — código do repositório
import subprocess, shutil, json, os
from pathlib import Path
from collections import Counter
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/optimistic-fermat-k8kggg",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["pip", "install", "-q", "numpy"], check=True)
print("repositório pronto em", REPO, flush=True)

In [ ]:
# Célula 1 — conferir os exemplos gerados (Drive)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
EXEMPLOS = Path("/content/drive/MyDrive/CRIVO/compreensao/exemplos_candidatos.jsonl")
if not EXEMPLOS.is_file():
    raise SystemExit("Não achei exemplos_candidatos.jsonl na pasta CRIVO/compreensao do Drive.")
L = [json.loads(l) for l in EXEMPLOS.read_text(encoding="utf-8").splitlines()]
print("total:", len(L))
print("status:", Counter(x["status"] for x in L))
print("tipo:", Counter(x["tipo"] for x in L))
print("split:", Counter(x["split"] for x in L))

In [ ]:
# Célula 2 — medir o Crivo atual nas réguas (não altera nada)
r1 = subprocess.run(["python", "avaliacoes/regua_compreensao_v1/rodar.py", "/content/resultado_compreensao.json"],
                    capture_output=True, text=True)
print("régua de compreensão (55):")
print("\n".join(l for l in r1.stdout.splitlines() if l.startswith("RESUMO")) or r1.stderr[-800:])
env = dict(os.environ, REGUA_CASOS=str(REPO / "avaliacoes/regua_frequentes_v1/casos.json"))
r2 = subprocess.run(["python", "avaliacoes/regua_compreensao_v1/rodar.py", "/content/resultado_frequentes.json"],
                    capture_output=True, text=True, env=env)
print("régua de 100 perguntas frequentes:")
print("\n".join(l for l in r2.stdout.splitlines() if l.startswith("RESUMO")) or r2.stderr[-800:])

Quando você tiver os números, eles são o ponto de partida. O próximo passo é o treino, que vai ser
medido contra exatamente essas réguas. Não cole aqui nenhuma senha ou dado pessoal.